#**Business Understanding**

##Problem Statement

Clinical trials frequently experience participant dropout, which can compromise statistical power, increase study costs, and delay regulatory approval.

Early identification of participants at risk of dropping out could help research teams implement retention strategies and improve trial completion rates.

##Machine Learning Objective

Develop a binary classification model capable of predicting whether a participant will drop out of a clinical trial using demographic and clinical information collected during the study.

###1) Conect to Drive

In [2]:
#Conect to drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.listdir("/content/drive/MyDrive/Bioinformatica/Data Science/Clinical_trial_dropout_prediction/data/raw")


Mounted at /content/drive


['synthetic_clinical_trial_data.csv']

###2) Import Libraries and Load Dataset

In [3]:
import pandas as pd

#Load Dataset
df = pd.read_csv("/content/drive/MyDrive/Bioinformatica/Data Science/Clinical_trial_dropout_prediction/data/raw/synthetic_clinical_trial_data.csv")

###3) Initial Inspection


In [4]:
#View of dataset
df.head(10)

,Subject_ID,Site_ID,Age,Gender,Enrollment_Date,Treatment_Group,Adverse_Events,Dropout,Systolic_BP,Diastolic_BP,Cholesterol_Level
0,1,49,54,Male,1/1/22,Drug A,0,0,117,74,229
1,2,37,44,Male,1/2/22,Placebo,1,0,111,57,173
2,3,1,58,Male,1/3/22,Drug A,0,1,122,89,220
3,4,25,48,Male,1/4/22,Drug B,0,0,122,85,175
4,5,10,57,Female,1/5/22,Drug A,2,0,105,90,185
5,6,36,59,Male,1/6/22,Placebo,1,0,128,85,206
6,7,18,44,Male,1/7/22,Drug A,3,1,139,120,197
7,8,49,45,Female,1/8/22,Drug B,1,1,141,75,172
8,9,47,40,Male,1/9/22,Drug A,0,0,103,84,197
9,10,24,56,Female,1/10/22,Drug A,3,0,119,93,215


In [5]:
# Random Sample
df.sample(10)

,Subject_ID,Site_ID,Age,Gender,Enrollment_Date,Treatment_Group,Adverse_Events,Dropout,Systolic_BP,Diastolic_BP,Cholesterol_Level
333,334,44,56,Female,11/30/22,Drug A,0,0,122,91,153
766,767,18,43,Female,2/6/24,Placebo,0,0,135,66,172
530,531,35,61,Female,6/15/23,Drug A,3,0,115,85,226
755,756,49,54,Female,1/26/24,Drug B,2,0,94,83,182
990,991,39,41,Female,9/17/24,Drug B,5,1,112,80,213
385,386,46,31,Female,1/21/23,Placebo,0,0,109,87,196
577,578,20,54,Male,8/1/23,Drug A,1,0,114,85,228
666,667,43,67,Female,10/29/23,Drug A,1,1,93,90,194
432,433,33,34,Male,3/9/23,Drug B,0,1,108,84,157
301,302,7,69,Female,10/29/22,Drug A,0,0,157,73,199


In [6]:
# Dataset dimentions
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

# Column information
df.info()


Rows: 1000
Columns: 11
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Subject_ID         1000 non-null   int64 
 1   Site_ID            1000 non-null   int64 
 2   Age                1000 non-null   int64 
 3   Gender             1000 non-null   object
 4   Enrollment_Date    1000 non-null   object
 5   Treatment_Group    1000 non-null   object
 6   Adverse_Events     1000 non-null   int64 
 7   Dropout            1000 non-null   int64 
 8   Systolic_BP        1000 non-null   int64 
 9   Diastolic_BP       1000 non-null   int64 
 10  Cholesterol_Level  1000 non-null   int64 
dtypes: int64(8), object(3)
memory usage: 86.1+ KB


In [10]:
# Numerical and statistical summary
pd.set_option("display.precision", 2)
df.describe()

,Subject_ID,Site_ID,Age,Adverse_Events,Dropout,Systolic_BP,Diastolic_BP,Cholesterol_Level
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,500.50,26.70,50.45,1.05,0.16,119.54,80.19,200.34
std,288.82,14.10,15.38,1.42,0.37,15.82,10.58,31.74
min,1.00,1.00,6.00,0.00,0.00,73.00,47.00,83.00
25%,250.75,15.00,39.00,0.00,0.00,109.00,73.00,179.00
50%,500.50,27.00,51.00,1.00,0.00,119.00,80.00,200.00
75%,750.25,38.25,61.00,2.00,0.00,129.00,87.00,220.00
max,1000.00,50.00,98.00,15.00,1.00,200.00,120.00,350.00


In [ ]:
# Categorical summary
df.describe(include="object")

,Gender,Enrollment_Date,Treatment_Group
count,1000,1000,1000
unique,2,1000,3
top,Female,9/26/24,Placebo
freq,503,1,363


###4) Data Quality Assessment

In [11]:
# Duplicated
print(f"Duplicated rows: {df.duplicated().sum()}")


Duplicated rows: 0


In [12]:
# Nulls
print("Missing values per column:")
df.isnull().sum()


Missing values per column:


,0
Subject_ID,0
Site_ID,0
Age,0
Gender,0
Enrollment_Date,0
Treatment_Group,0
Adverse_Events,0
Dropout,0
Systolic_BP,0
Diastolic_BP,0


In [13]:
# Objective variable
print("Traget distribution:")
df["Dropout"].value_counts()
df["Dropout"].value_counts(normalize=True)   #Does imbalance exist?


Traget distribution:


,proportion
Dropout,
0,0.84
1,0.16


In [15]:
# Subject ID
df["Subject_ID"].nunique()

1000

In [16]:
# Site ID
df["Site_ID"].nunique()

50

###5) Variable Classification

| Variable | Pandas dtype | Statistical Type | ML Role |
|----------|--------------|------------------|----------|
| Subject_ID | int64 | Identifier | Exclude |
| Site_ID | int64 | Nominal categorical | Predictor |
| Age | int64 | Continuous numerical | Predictor |
| Gender | object | Nominal categorical | Predictor |
| Enrollment_Date | object | Date (stored as object) | Predictor |
| Treatment_Group | object | Nominal categorical | Predictor |
| Adverse_Events | int64 | Discrete count | Predictor |
| Systolic_BP | int64 | Continuous numerical | Predictor |
| Diastolic_BP | int64 | Continuous numerical | Predictor |
| Cholesterol_Level | int64 | Continuous numerical | Predictor |
| Dropout | int64 | Binary categorical | Target |

###6) Initial Findings

**Dataset Structure**

*   The dataset contains **1,000** observations and **11** variables.
*   Eight variables are numerical (int64) and three are categorical (object).

**Target Variable**

*   The target variable is Dropout.
*   Approximately **16%** of participants dropped out of the clinical trial, indicating a moderate class imbalance.

**Data Quality**

*   No duplicated records were detected.
*   No missing values were identified.
*   Overall, the dataset presents good initial data quality.

**Variables**

*   `Subject_ID` is a unique identifier and will likely be excluded from model training.
*   `Site_ID` may contain useful information regarding the clinical center and will be explored further during EDA, it contains **50** differents IDs.
*   `Enrollment_Date` is currently stored as text and should be converted to a datetime format during preprocessing.

**Next Steps**

*   Explore the distribution of numerical and categorical variables.
*   Investigate potential outliers (e.g., unusually young participants).
*   Analyze relationships between predictor variables and patient dropout.